In [1]:
import sqlite3
import pandas as pd
import os

print("SQLite is ready!")

SQLite is ready!


In [2]:
db_path = "../sql/ecommerce_analytics.db"

conn = sqlite3.connect(db_path)

print("Database created successfully!")
print(db_path)

Database created successfully!
../sql/ecommerce_analytics.db


In [3]:
df_clean = pd.read_csv(
    "../data/cleaned/online_retail_cleaned.csv",
    parse_dates=["InvoiceDate", "Month_Start"]
)

print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))

Rows: 1033036
Columns: 17


In [4]:
df_clean.to_sql(
    "transactions",
    conn,
    if_exists="replace",
    index=False
)

print("transactions table created!")

transactions table created!


In [5]:
query = """
SELECT COUNT(*) AS total_rows
FROM transactions;
"""

pd.read_sql_query(query, conn)

,total_rows
0,1033036


In [6]:
query = """
SELECT 
    SUM(Revenue) AS Total_Revenue
FROM transactions
WHERE Is_Return = 0
  AND Quantity > 0
  AND Price > 0;
"""

pd.read_sql_query(query, conn)

,Total_Revenue
0,2.047626e+07


In [7]:
query = """
SELECT 
    COUNT(DISTINCT Invoice) AS Total_Orders
FROM transactions
WHERE Is_Return = 0
  AND Quantity > 0
  AND Price > 0;
"""

pd.read_sql_query(query, conn)

,Total_Orders
0,40077


In [8]:
query = """
SELECT 
    COUNT(DISTINCT Customer_ID) AS Total_Customers
FROM transactions
WHERE Customer_ID IS NOT NULL
  AND Is_Return = 0
  AND Quantity > 0
  AND Price > 0;
"""

pd.read_sql_query(query, conn)

,Total_Customers
0,5878


In [9]:
query = """
SELECT
    Country,
    ROUND(SUM(Revenue), 2) AS Total_Revenue
FROM transactions
WHERE Is_Return = 0
  AND Quantity > 0
  AND Price > 0
GROUP BY Country
ORDER BY Total_Revenue DESC
LIMIT 10;
"""

country_sales = pd.read_sql_query(query, conn)

country_sales

,Country,Total_Revenue
0,United Kingdom,17410196.12
1,EIRE,658767.31
2,Netherlands,554038.09
3,Germany,425019.71
4,France,350456.09
5,Australia,169283.46
6,Spain,108332.49
7,Switzerland,100685.59
8,Sweden,91869.82
9,Denmark,68580.69


In [10]:
query = """
SELECT
    Description,
    SUM(Quantity) AS Units_Sold,
    ROUND(SUM(Revenue), 2) AS Revenue
FROM transactions
WHERE Is_Return = 0
  AND Quantity > 0
  AND Price > 0
GROUP BY Description
ORDER BY Revenue DESC
LIMIT 10;
"""

top_products_sql = pd.read_sql_query(query, conn)

top_products_sql

,Description,Units_Sold,Revenue
0,Manual,9634,339241.29
1,REGENCY CAKESTAND 3 TIER,26478,330590.32
2,DOTCOM POSTAGE,1415,309854.11
3,WHITE HANGING HEART T-LIGHT HOLDER,94658,260990.22
4,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60
5,PARTY BUNTING,28200,148318.28
6,JUMBO BAG RED RETROSPOT,77699,148073.47
7,ASSORTED COLOUR BIRD ORNAMENT,80082,129324.49
8,POSTAGE,5363,125682.42
9,PAPER CHAIN KIT 50'S CHRISTMAS,35084,117760.29


In [11]:
query = """
SELECT
    CASE
        WHEN Revenue < 20 THEN 'Low Value'
        WHEN Revenue < 100 THEN 'Medium Value'
        WHEN Revenue < 500 THEN 'High Value'
        ELSE 'Very High Value'
    END AS Order_Category,
    
    COUNT(*) AS Transactions,
    ROUND(SUM(Revenue), 2) AS Total_Revenue

FROM transactions

WHERE Is_Return = 0
  AND Quantity > 0
  AND Price > 0

GROUP BY Order_Category
ORDER BY Total_Revenue DESC;
"""

order_categories = pd.read_sql_query(query, conn)

order_categories

,Order_Category,Transactions,Total_Revenue
0,Low Value,811581,6830056.95
1,Medium Value,169874,6420056.53
2,High Value,24187,4430162.24
3,Very High Value,2271,2795984.73


In [12]:
query = """
SELECT
    Country,
    ROUND(SUM(Revenue), 2) AS Revenue
FROM transactions
WHERE Is_Return = 0
  AND Quantity > 0
  AND Price > 0
GROUP BY Country
HAVING SUM(Revenue) > 100000
ORDER BY Revenue DESC;
"""

pd.read_sql_query(query, conn)

,Country,Revenue
0,United Kingdom,17410196.12
1,EIRE,658767.31
2,Netherlands,554038.09
3,Germany,425019.71
4,France,350456.09
5,Australia,169283.46
6,Spain,108332.49
7,Switzerland,100685.59


In [13]:
query = """
WITH monthly_sales AS (

    SELECT
        Month_Start,
        SUM(Revenue) AS Revenue

    FROM transactions

    WHERE Is_Return = 0
      AND Quantity > 0
      AND Price > 0

    GROUP BY Month_Start
)

SELECT
    Month_Start,
    ROUND(Revenue, 2) AS Revenue

FROM monthly_sales

ORDER BY Month_Start;
"""

monthly_sql = pd.read_sql_query(query, conn)

monthly_sql

,Month_Start,Revenue
0,2009-12-01 00:00:00,822483.95
1,2010-01-01 00:00:00,651155.11
2,2010-02-01 00:00:00,551504.73
3,2010-03-01 00:00:00,830915.26
4,2010-04-01 00:00:00,678875.25
5,2010-05-01 00:00:00,657705.50
6,2010-06-01 00:00:00,749537.31
7,2010-07-01 00:00:00,648810.27
8,2010-08-01 00:00:00,695251.91
9,2010-09-01 00:00:00,921696.99


In [14]:
query = """
WITH country_sales AS (

    SELECT
        Country,
        SUM(Revenue) AS Revenue

    FROM transactions

    WHERE Is_Return = 0
      AND Quantity > 0
      AND Price > 0

    GROUP BY Country
)

SELECT
    Country,
    ROUND(Revenue, 2) AS Revenue,

    RANK() OVER (
        ORDER BY Revenue DESC
    ) AS Revenue_Rank

FROM country_sales

ORDER BY Revenue_Rank;
"""

country_rank = pd.read_sql_query(query, conn)

country_rank.head(10)

,Country,Revenue,Revenue_Rank
0,United Kingdom,17410196.12,1
1,EIRE,658767.31,2
2,Netherlands,554038.09,3
3,Germany,425019.71,4
4,France,350456.09,5
5,Australia,169283.46,6
6,Spain,108332.49,7
7,Switzerland,100685.59,8
8,Sweden,91869.82,9
9,Denmark,68580.69,10


In [15]:
query = """
WITH monthly_sales AS (

    SELECT
        Month_Start,
        SUM(Revenue) AS Revenue

    FROM transactions

    WHERE Is_Return = 0
      AND Quantity > 0
      AND Price > 0

    GROUP BY Month_Start
)

SELECT
    Month_Start,
    ROUND(Revenue, 2) AS Revenue,

    ROUND(
        LAG(Revenue) OVER (
            ORDER BY Month_Start
        ),
        2
    ) AS Previous_Month_Revenue,

    ROUND(
        (
            Revenue -
            LAG(Revenue) OVER (
                ORDER BY Month_Start
            )
        )
        /
        NULLIF(
            LAG(Revenue) OVER (
                ORDER BY Month_Start
            ),
            0
        ) * 100,
        2
    ) AS MoM_Growth_Percentage

FROM monthly_sales

ORDER BY Month_Start;
"""

monthly_growth = pd.read_sql_query(query, conn)

monthly_growth

,Month_Start,Revenue,Previous_Month_Revenue,MoM_Growth_Percentage
0,2009-12-01 00:00:00,822483.95,NaN,NaN
1,2010-01-01 00:00:00,651155.11,822483.95,-20.83
2,2010-02-01 00:00:00,551504.73,651155.11,-15.30
3,2010-03-01 00:00:00,830915.26,551504.73,50.66
4,2010-04-01 00:00:00,678875.25,830915.26,-18.30
5,2010-05-01 00:00:00,657705.50,678875.25,-3.12
6,2010-06-01 00:00:00,749537.31,657705.50,13.96
7,2010-07-01 00:00:00,648810.27,749537.31,-13.44
8,2010-08-01 00:00:00,695251.91,648810.27,7.16
9,2010-09-01 00:00:00,921696.99,695251.91,32.57


In [16]:
query = """
WITH customer_sales AS (

    SELECT
        Customer_ID,
        SUM(Revenue) AS Revenue

    FROM transactions

    WHERE Customer_ID IS NOT NULL
      AND Is_Return = 0
      AND Quantity > 0
      AND Price > 0

    GROUP BY Customer_ID
)

SELECT
    Customer_ID,
    ROUND(Revenue, 2) AS Revenue,

    DENSE_RANK() OVER (
        ORDER BY Revenue DESC
    ) AS Customer_Rank

FROM customer_sales

ORDER BY Customer_Rank

LIMIT 20;
"""

top_customers_sql = pd.read_sql_query(query, conn)

top_customers_sql

,Customer_ID,Revenue,Customer_Rank
0,18102.0,580987.04,1
1,14646.0,528602.52,2
2,14156.0,313437.62,3
3,14911.0,291420.81,4
4,17450.0,244784.25,5
5,13694.0,195640.69,6
6,17511.0,172132.87,7
7,16446.0,168472.50,8
8,16684.0,147142.77,9
9,12415.0,144458.37,10


In [17]:
rfm = pd.read_csv(
    "../data/cleaned/customer_rfm.csv"
)

rfm.to_sql(
    "customer_rfm",
    conn,
    if_exists="replace",
    index=False
)

print("RFM table created!")

RFM table created!


In [18]:
query = """
SELECT
    Customer_Segment,
    COUNT(*) AS Customers,
    ROUND(SUM(Monetary), 2) AS Revenue,
    ROUND(AVG(Monetary), 2) AS Avg_Spend,
    ROUND(AVG(Frequency), 2) AS Avg_Frequency,
    ROUND(AVG(Recency), 2) AS Avg_Recency

FROM customer_rfm

GROUP BY Customer_Segment

ORDER BY Revenue DESC;
"""

rfm_sql = pd.read_sql_query(query, conn)

rfm_sql

,Customer_Segment,Customers,Revenue,Avg_Spend,Avg_Frequency,Avg_Recency
0,Champions,1289,11819626.88,9169.61,17.15,18.72
1,Loyal Customers,1413,2713749.35,1920.56,5.46,70.13
2,At Risk,825,1590384.58,1927.74,4.94,367.88
3,Lost Customers,1526,655444.68,429.52,1.25,457.75
4,New Customers,441,391838.26,888.52,1.46,26.97
5,Potential Loyalists,384,203760.52,530.63,1.36,105.22


In [19]:
country_sales.to_csv(
    "../data/cleaned/sql_country_sales.csv",
    index=False
)

top_products_sql.to_csv(
    "../data/cleaned/sql_top_products.csv",
    index=False
)

monthly_growth.to_csv(
    "../data/cleaned/sql_monthly_growth.csv",
    index=False
)

top_customers_sql.to_csv(
    "../data/cleaned/sql_top_customers.csv",
    index=False
)

rfm_sql.to_csv(
    "../data/cleaned/sql_rfm_segments.csv",
    index=False
)

print("SQL analysis results saved!")

SQL analysis results saved!


In [20]:
conn.close()

print("Database connection closed.")

Database connection closed.
